# Iowa Liquor Sales: Merchandising Experiment Proposal

**Portfolio refresh of a 2019 Thinkful project**

This refresh preserves the original business idea while correcting data semantics, reconstructing the historical window, adding data-quality checks, and redesigning the proposal as a randomized store-level experiment.

> The original 2019 notebook remains unchanged for provenance. This notebook is an experiment proposal and historical audit, not a claim that an experiment was run.


## Historical data provenance

Preserved output from the original notebook indicates a historical Iowa Liquor Sales snapshot covering **January 3, 2012 through October 31, 2017**, with **12,590,909 rows** (final zero-based index `12,590,908`).

The current Iowa Data Hub describes the records as spirits purchases by licensed retailers. Its modern catalog uses `ordered_on` as the record date and `invoice_id` as the unique transaction-line identifier. A current export filtered to the same dates is therefore a **reconstruction of the historical window**; retrospective corrections can prevent an exact row-for-row match with the 2017 snapshot.

Official catalog: https://data.iowa.gov/catalog/dataset/1051

### Critical semantic correction

The original notebook calculated `Sale (Dollars) - State Bottle Cost × Bottles Sold` and called it store **Profit**. The Iowa data do not contain Casey's consumer revenue or operating costs, so retailer profit cannot be calculated from this source. The refreshed analysis uses observable wholesale-demand measures such as **bottles ordered** and **order dollars**.


## 1. Load and reconstruct the historical window

The raw file is not committed because the historical window contains about 12.6 million rows. Use either the exact historical snapshot, if available, or a current Iowa Data Hub export. Save it at `data/iowa_liquor_sales.csv` or change `DATA_PATH`.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

START_DATE = pd.Timestamp("2012-01-03")
END_DATE = pd.Timestamp("2017-10-31")
HISTORICAL_ROW_REFERENCE = 12_590_909
DATA_PATH = Path("data/iowa_liquor_sales.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"{DATA_PATH} not found. Download Iowa Liquor Sales from the official "
        "Iowa Data Hub, save it locally, or update DATA_PATH."
    )

raw = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Rows in supplied file: {len(raw):,}")


## 2. Normalize legacy and modern schemas

The original export and the modern Iowa Data Hub use different column labels. The mapping below accepts both where known, including modern `ordered_on` and `invoice_id`.


In [ ]:
def normalize_columns(df):
    aliases = {
        "Invoice/Item Number": "invoice_item_number",
        "invoice_and_item_number": "invoice_item_number",
        "invoice_id": "invoice_item_number",
        "Date": "date", "date": "date", "ordered_on": "date",
        "Store Number": "store_number", "store_number": "store_number",
        "Store Name": "store_name", "store_name": "store_name",
        "Item Number": "item_number", "item_number": "item_number",
        "Item Description": "item_description", "item_description": "item_description",
        "Category Name": "category_name", "category_name": "category_name",
        "State Bottle Cost": "state_bottle_cost", "state_bottle_cost": "state_bottle_cost",
        "State Bottle Retail": "state_bottle_retail", "state_bottle_retail": "state_bottle_retail",
        "Bottles Sold": "bottles_ordered", "bottles_sold": "bottles_ordered",
        "Sale (Dollars)": "order_dollars", "sale_dollars": "order_dollars",
        "Volume Sold (Liters)": "volume_liters", "volume_sold_liters": "volume_liters",
    }
    return df.rename(columns={c: aliases[c] for c in df.columns if c in aliases}).copy()

df = normalize_columns(raw)
required = {
    "date", "store_number", "store_name", "item_number",
    "state_bottle_cost", "state_bottle_retail",
    "bottles_ordered", "order_dollars"
}
missing = required - set(df.columns)
if missing:
    raise KeyError(f"Required columns missing after normalization: {sorted(missing)}")

df["date"] = pd.to_datetime(df["date"], errors="coerce")
historical = df[df["date"].between(START_DATE, END_DATE, inclusive="both")].copy()

print("Observed date range:", historical["date"].min(), "to", historical["date"].max())
print(f"Reconstructed rows: {len(historical):,}")
print(f"Preserved snapshot reference: {HISTORICAL_ROW_REFERENCE:,}")
print(f"Difference: {len(historical) - HISTORICAL_ROW_REFERENCE:+,}")
if "invoice_item_number" in historical:
    print("Duplicate invoice-item IDs:", historical["invoice_item_number"].duplicated().sum())


## 3. Validate order arithmetic

The original notebook interpreted zero and negative derived values as possible store losses. Here those rows are treated first as data-quality/correction records. A useful check is whether `order_dollars ≈ state_bottle_retail × bottles_ordered`.


In [ ]:
for col in ["state_bottle_cost", "state_bottle_retail", "bottles_ordered", "order_dollars", "volume_liters"]:
    if col in historical:
        historical[col] = pd.to_numeric(
            historical[col].astype(str)
            .str.replace("$", "", regex=False)
            .str.replace(",", "", regex=False),
            errors="coerce"
        )

historical["expected_order_dollars"] = (
    historical["state_bottle_retail"] * historical["bottles_ordered"]
)
historical["order_reconciles"] = np.isclose(
    historical["order_dollars"],
    historical["expected_order_dollars"],
    rtol=0, atol=0.01, equal_nan=False
)

quality_summary = pd.Series({
    "rows": len(historical),
    "zero_or_negative_bottles": (historical["bottles_ordered"] <= 0).sum(),
    "zero_or_negative_order_dollars": (historical["order_dollars"] <= 0).sum(),
    "non_reconciling_order_rows": (~historical["order_reconciles"]).sum(),
})
quality_summary


## 4. Casey's demand and product opportunity

The original project focused on Casey's General Store. The refresh keeps that business context, uses stable `store_number` where possible, and ranks products by **item number** rather than item/category pairs that can duplicate the same item when labels change.


In [ ]:
analysis = historical[
    historical["date"].notna()
    & historical["store_number"].notna()
    & historical["item_number"].notna()
    & (historical["bottles_ordered"] > 0)
    & (historical["order_dollars"] > 0)
].copy()

casey = analysis[
    analysis["store_name"].astype(str).str.contains("Casey", case=False, na=False)
].copy()

lookup_cols = {}
if "item_description" in casey:
    lookup_cols["item_description"] = ("item_description", "last")
if "category_name" in casey:
    lookup_cols["category_name"] = ("category_name", "last")

product_rank = (
    casey.groupby("item_number")
    .agg(
        bottles_ordered=("bottles_ordered", "sum"),
        order_dollars=("order_dollars", "sum"),
        active_stores=("store_number", "nunique"),
        order_lines=("item_number", "size"),
    )
    .sort_values("bottles_ordered", ascending=False)
)

if lookup_cols:
    lookup = casey.sort_values("date").groupby("item_number").agg(**lookup_cols)
    product_rank = product_rank.join(lookup)

product_rank.head(15)


In [ ]:
top_products = product_rank.head(10).sort_values("bottles_ordered")
ax = top_products["bottles_ordered"].plot(kind="barh", figsize=(9, 6))
ax.set(title="Top Casey's Items by Bottles Ordered",
       xlabel="Bottles ordered", ylabel="Item number")
plt.tight_layout()
plt.show()

original_candidate_items = {"11788", "11776", "35918"}
product_rank.loc[product_rank.index.astype(str).isin(original_candidate_items)]


### What changed from the original product ranking

The 2019 notebook hard-coded items `11788`, `11776`, and `35918` after ranking a mislabeled margin metric. The refreshed ranking may differ because it uses observable demand, groups first by stable item ID, and may be run against a modern reconstruction containing retrospective corrections.


## 5. Store-week baseline for experiment planning

Because merchandising is implemented at the store level, the experiment should randomize and analyze at the store level. Historical orders can estimate baseline activity and variability, but cannot create a causal treatment effect.


In [ ]:
casey["week"] = casey["date"].dt.to_period("W").dt.start_time

store_week = (
    casey.groupby(["store_number", "week"])
    .agg(
        bottles_ordered=("bottles_ordered", "sum"),
        order_dollars=("order_dollars", "sum"),
        order_lines=("item_number", "size"),
    )
    .reset_index()
)

store_baseline = store_week.groupby("store_number").agg(
    active_weeks=("week", "nunique"),
    mean_weekly_bottles=("bottles_ordered", "mean"),
    sd_weekly_bottles=("bottles_ordered", "std"),
    mean_weekly_order_dollars=("order_dollars", "mean"),
)
store_baseline.describe()


## 6. Portfolio-ready A/B test design

### Business question
**Does standardized in-store merchandising for selected liquor products increase product demand at Casey's stores relative to comparable stores without the treatment?**

### Design
- **Population:** Casey's stores with stable recent activity and enough baseline observations.
- **Assignment unit:** store.
- **Treatment:** one standardized merchandising treatment, such as defined shelf placement plus a standardized sign.
- **Control:** business as usual.
- **Randomization:** 50/50 treatment/control, preferably stratified or matched on recent store volume and, if sample size permits, geography.
- **Pre-period:** approximately 6–8 weeks for baseline measurement and power estimation.
- **Test period:** approximately 4 weeks, finalized from power analysis and operating constraints.

### Metrics
- **Preferred primary outcome:** consumer POS units of promoted items per store-week.
- **Fallback using Iowa data:** bottles ordered of promoted items per store-week, explicitly labeled a lagging wholesale-demand proxy.
- **Secondary outcomes:** promoted-item order dollars, total liquor volume/order dollars, order frequency, and non-promoted-item volume to detect cannibalization.

### Hypotheses
- **H0:** treatment does not change mean promoted-item demand relative to control.
- **H1:** treatment increases mean promoted-item demand relative to control.

### Analysis
Estimate treatment lift at the store assignment level. A pre/post difference-in-differences contrast can improve precision:

`(Treatment post − Treatment pre) − (Control post − Control pre)`

Report effect size and confidence interval using a pre-specified alpha level. Define the minimum detectable effect, power target, duration, and analysis date **before launch**; do not use informal two-week early stopping.


## 7. Audit conclusions and limitations

The refresh preserves the original business instinct but corrects the measurement and causal design:

- Iowa records wholesale purchases by licensees, not Casey's consumer sales.
- The original `Profit` field is not Casey's profit.
- Zero/non-reconciling rows require data-quality review before business interpretation.
- Stable store/item identifiers are preferable to display names/categories.
- The original proposal lacked a control group and random assignment, so it was not a true A/B test.
- Historical data are useful for product discovery, store eligibility, and baseline variance, not for manufacturing causal treatment results.
- A real experiment should use recent baseline data and, ideally, Casey's POS outcomes.

The exact 2017 raw file is not bundled. The preserved original notebook's **12,590,909 rows through October 31, 2017** remain the historical checkpoint.

### Project files
- `Iowa_Liquor_Store_AB_Test_Proposal.ipynb` — preserved original
- `Iowa_Liquor_Store_AB_Test_Audit.md` — audit trail
- `Iowa_Liquor_AB_Test_Portfolio.ipynb` — this refreshed notebook
